# Citrus: final 50,000 draft candidates
Resume the completed 100,000-record checkpoint. This notebook increases the cumulative target to 150,000 (45,000 knowledge / 105,000 practice), retaining previous candidates and normalized exact duplicate checks. It generates only the remaining 50,000.
Use a CUDA GPU and Internet. Multiple sessions may be required. All output remains draft; generation does not train a model or certify factual accuracy.


In [ ]:
%pip install -q "transformers>=4.51,<5" "accelerate>=1,<2" "bitsandbytes>=0.45,<1" sentencepiece


In [ ]:
import pathlib, urllib.request
# Fixed repository revision makes the notebook reproducible.
REVISION = "3d5e8bbae98d7bef517d558611e79da2fec2414a"
BASE = f"https://raw.githubusercontent.com/gentle58715-ship-it/Citrus-Training-/{REVISION}"
ROOT = pathlib.Path("/kaggle/working/citrus-generator")
ROOT.mkdir(exist_ok=True)
for name in ["generate_citrus_kaggle.py", "audit_citrus_v3.py"]:
    urllib.request.urlretrieve(f"{BASE}/scripts/{name}", ROOT / name)


In [ ]:
import sqlite3, json
OUT = pathlib.Path("/kaggle/working/citrus-150k")
OUT.mkdir(exist_ok=True)
# REQUIRED on first run: attach completed 100k output to Kaggle.
# For later sessions, attach the latest 150k-run checkpoint instead.
RESUME_DB = ""
if not (OUT / "progress.sqlite").exists():
    if not RESUME_DB:
        raise RuntimeError("Set RESUME_DB to the attached progress.sqlite; do not start batch 3 empty")
    src = sqlite3.connect(f"file:{RESUME_DB}?mode=ro", uri=True)
    counts = dict(src.execute("SELECT kind, COUNT(*) FROM records GROUP BY kind"))
    if counts.get("knowledge", 0) < 30000 or counts.get("practice", 0) < 70000:
        src.close()
        raise RuntimeError("First 100,000 is not complete. Resume the second notebook before starting batch 3")
    if counts.get("knowledge", 0) > 45000 or counts.get("practice", 0) > 105000:
        src.close()
        raise RuntimeError("Checkpoint already exceeds this notebook's 150k quotas")
    dst = sqlite3.connect(OUT / "progress.sqlite")
    src.backup(dst)
    dst.close(); src.close()


In [ ]:
import subprocess
MAX_MINUTES = 480
BASE_ARGS = ["python3", "-u", str(ROOT / "generate_citrus_kaggle.py"),
             "--out", str(OUT), "--knowledge", "45000", "--practice", "105000",
             "--extend-targets"]
try:
    subprocess.run(BASE_ARGS + ["--max-minutes", str(MAX_MINUTES)], check=True)
finally:
    subprocess.run(BASE_ARGS + ["--export-only"], check=True)


In [ ]:
import json, shutil
manifest = json.loads((OUT / "manifest.json").read_text())
print(json.dumps(manifest, ensure_ascii=False, indent=2))
subprocess.run(["python3", str(ROOT / "audit_citrus_v3.py"), "--root", str(OUT)], check=True)
archive = shutil.make_archive("/kaggle/working/Citrus-150k-progress", "zip", OUT)
print("Saved:", archive)


## Output and quality review
The final JSONL contains all three batches: 150,000 draft candidates total. Do not concatenate the first and second cumulative exports into it again. Existing records remain in the same SQLite database, with exact duplicate checks spanning all batches. Semantic duplication still requires separate review.
Generation completion is reported only when actual counts reach 45,000 knowledge and 105,000 practice. This does not mean approved data: every candidate remains draft and runtime_verified=false until actually reviewed/tested. Use the audit script to export approved examples and keep related task families in one split.
No Kaggle generation job has been launched by creating this notebook.
